# MTL Mode Demo

This notebook provides a demonstration of the parsing and verification processes for ML-DSA digital signatures within the context of Merkle Tree Ladder (MTL) Mode for DNSSEC. A detailed description of ML-DSA-MTL for DNSSEC can be found in draft-kaizer-dnsop-ml-dsa-mtl-dnssec-01.  

## Merkle Tree Ladder (MTL) Mode

MTL Mode is a cryptographic technique designed to reduce the size impact of post-quantum cryptographic (PQC) digital signatures on an evolving series of messages. Instead of signing every single message or record individually with a massive PQC algorithm, MTL mode aggregates messages into an evolving structure of binary hash trees (assuming a binary rung strategy). The underlying signature scheme is then applied only to a compact data structure called the ladder. MTL Mode consists of the following components: 
  * **Authentication Path**: An ordered sequence of sibling node hashes required to mathematically reconstruct a path from a specific message leaf up to its corresponding rung in the ladder.
  * **Index**: An integer value *i* that identifies a leaf node in a node set. Indexes start at 0 and are consecutively assigned.
  * **Index Pair**: A pair of integer values (*L*,*R*) that together identify a node in a node set based on the lowest and highest indexes of the consecutively-indexed leaf nodes that the node authenticates.  For a leaf node, the index pair is (*i*,*i*) which is used interchangeably with the leaf index *i*.
  * **Internal Node**: A node in a node set whose value is the hash of two child nodes.
  * **The MTL Ladder**: A collection of one or more rungs that can be used to verify an authentication path.
      * This demo employs the MTL Mode with a binary rung strategy. The selection of rungs for a given state corresponds directly to the binary representation of the total number of messages (*N*) currently in the node set.
      * Every active leaf node must have a valid hash transmission path to exactly one of the rungs in the current ladder. As *N* increments, the ladder structure dynamically shifts, merging smaller rungs into higher-level rungs.
  * **Leaf Node**: A node in a node set whose value is the hash of a single message.
  * **Message**: A set of bytes that are intended to be signed and later verified.
  * **Node Set**: A dynamic data structure maintained by the signer to represent an expanding sequence of authenticated data values.
      * Every message *m{i}* in the series is cryptographically processed into a leaf node. The index *i* continually increments as new messages are added. The leaf value is typically computed as *H*(prefix || *m{i}*), where *H* is a secure cryptographic hash function, and the prefix ensures cryptographic domain separation.
      * Internal nodes are generated by iteratively pairing and hashing adjacent lower-level nodes. Because the number of messages grows sequentially, the node set does not form a single, uniform balanced tree. Instead, it is an accumulating forest of perfect binary trees of varying heights, linked implicitly by their indices.
  * **Root Node**: A node in a node set that has no ancestors.
  * **Series Identifier (SID)**: A cryptographically unique value associated with an instance of a MTL node set.
  * **Signature**: the high-overhead, asymmetric cryptographic proof that binds the signer's identity to the integrity of the entire ladder.
  * **Rung**: A node from a node set that can be used to authenticate one or more leaf nodes within that node set.  A rung may be a root node.

The diagram below illustrates a 14-node MTL structure using the binary rung strategy, with rungs marked by asterisks (*). H denotes the internal node hash function, while the leaf node hash function is omitted.

```
                 (0,7)*
                   |
                   H
           /------/ \------\
         (0,3)           (4,7)           (8,11)*
           |               |               |
           H               H               H
       /--/ \--\       /--/ \--\       /--/ \--\
     (0,1)   (2,3)   (4,5)   (6,7)   (8,9)  (10,11) (12,13)*
       |       |       |       |       |       |       |
       H       H       H       H       H       H       H
      / \     / \     / \     / \     / \     / \     / \
     0   1   2   3   4   5   6   7   8   9  10  11  12  13
```

## Signature Scheme
The MTL Mode is intended primarily for use with PQC signature schemes where the reduction of operational impact can be significant given their relatively large signature sizes. 

This demo supports ML-DSA (Module-Lattice-Based Digital Signature Algorithm), a quantum-resistant digital signature scheme standardized by NIST [FIPS204]. ML-DSA, formerly known as CRYSTALS-Dilithium, bases its cryptographic security on the hardness of finding short vectors in module lattices (a mathematical hard problem). ML-DSA maintains robust non-repudiation and authentication without relying on prime factorization. 

This demo supports **ML-DSA (ML-DSA-44/65/87)** algorithms for ladder signatures, utilizing OpenSSL for key generation, signing, and verification. The implementation relies on a single security parameter, *n*, which should be at least 16, with typical deployment values configured at 16, 24, or 32.

## Hash Algorithm

This demo supports the use of **SHAKE** as the hash function for use in the MTL mode. Note that as per draft-kaizer-dnsop-ml-dsa-mtl-dnssec-01, the context string fed into the hash function to generate the leaf node hash should be **NULL**.

## Demo Outline

```
+-- [MTL Mode Demo]
    +-- [Importing Libraries]
    +-- [Demo Configuration]
        |   +-- [DNS Message For Signing]
        |   +-- [Signing and Hashing Configurations]
        |   +-- [Public Key]
        |   +-- [MTL Mode Full Signature]
    +-- [(Section 1) Generating DNSSEC Signed Data]
    +-- [(Section 2) Public Key]

    +-- [(Section 3) Parsing Full Signature]
    |   +-- [(Section 3.1) Obtaining Series Identifier (SID)]
    |   +-- [(Section 3.2) Obtianing Authentication Path]
    |   +-- [(Section 3.3) Obtaining Ladder and Ladder Signature]
    |   +-- [(Section 3.4) Visualizing MTL Structure]

    +-- [(Section 4) Verification]
    |   +-- [(Section 4.1) - Validating Ladder Signature Using Public Key]
    |   +-- [(Section 4.2) - Validating Authentication Path]
        |   +-- [(Section 4.2.1) Recomputing Leaf Hash]
        |   +-- [(Section 4.2.2) Iteratively Hashing with Sibling Nodes for Internal Nodes]
        |   +-- [(Section 4.2.3) Target Match with Extracted Ladder Rung]
    +-- [End of Demo]
```

## Importing Libraries to Run This Demo: 

In [20]:
import struct
import socket
import os
import ctypes

Helper functions are imported right before execution. 

## Demo Configurations

### (1) DNS Message For Signing 
This message is a set of bytes that are intended to be signed and later verified.
The following values can be replaced with other DNS RRSIG RDATA and RRset data.

In [21]:
# RRSIG RDATA 
rrsig_meta = {
    "type_covered": 1,                                 # DNS record type A
    "algorithm": 131,                                  # Algorithm
    "labels": 3,                                       # "ns.example.org." has three labels
    "original_ttl": 3600,                              # The TTL the signature is based on
    "sig_expiration": 20260203132321,                  # Signature expiration timestamp
    "sig_inception": 20260106132321,                   # Signature inception timestamp
    "key_tag": 46491,                                  # ID of the DNSKEY used for signing
    "signer_name": "example.org."                      # Signer's name
}

# RRset Data
rrset = [
    {
        "name": "ns.example.org.", 
        "type": 1, 
        "class": 1, 
        "rdata": socket.inet_aton("192.168.1.50")
    },
]

### (2) Signing and Hashing Configurations
The following parameters can be modified. The signature algorithm must match the public key and signature specified below, while the hash algorithm and security parameters must match the ladder configuration. Note that **field sizes in the following sections depend on the signature algorithms and may vary if other algorithms are used, requiring corresponding modifications to the underlying functions**.

In [22]:
# ML-DSA signature algorithm (should be supported by OpenSSL version 3.5.0):
sig_alg = "ML-DSA-44"

# Hash algorithm for generating node hashes: this demo supports "shake".
hash_alg = "shake"

# Security parameter n: this demo supports n = 16, 24, or 32.
n = 16

### (3) Public Key
The content in data/mldsa_pubkey.txt or data/slhdsa_pubkey.txt can be modified with other public key values that correspond to the signature. 

In [23]:
# Read the corresponding public key according to the specified algorithm
if sig_alg.startswith("ML-DSA"):
    with open("data/mldsa_pubkey.txt", "r") as f:
        readin_key = f.read()
else:
    raise ValueError(f"Unsupported algorithm {sig_alg}.")
    
# The key stored in .txt is supposed to be in hex format
pubkey = bytes.fromhex(readin_key)

### (4) MTL Mode Signature
The content in data/mldsa_fullsig.txt can be modified with other values that contain the ladder and ladder signature. The ladder signature is supposed to be generated elsewhere outside this demo using the corresponding private key, and should match the public key specified above. 

In [24]:
# Read the corresponding signature according to the specified algorithm
if sig_alg.startswith("ML-DSA"):
    with open("data/mldsa_fullsig.txt", "r") as f:
        readin_fullsig = f.read()
else:
    raise ValueError(f"Unsupported algorithm {sig_alg}.")

# The signature stored in full_sig.txt is supposed to be in hex format
fullsig = bytes.fromhex(readin_fullsig)

## 1. Generating DNSSEC Signed Data 

#### Creating DNSSEC Signed Data Given RRSIG RDATA and RRset Data
The calling function concatenates the RRSIG RDATA (excluding the signature field) with the canonicalized RRset to create the signed data for DNSSEC. 
The signed data is later used to generate the leaf node for the authentication path verification in Seciton 4.2.1. 

In [25]:
from helper.utils import create_dnssec_signed_data

message = create_dnssec_signed_data(rrsig_meta, rrset)

print("─" * 50)
print(f"DNSSEC Signed Data:\n{message.hex(" ", -4).upper()}")
print("-" * 30)
print(f"Length of DNSSEC Signed Data: {len(message)} bytes")
print("─" * 50)

──────────────────────────────────────────────────
DNSSEC Signed Data:
00018303 00000E10 6981F6C9 695D0CC9 B59B0765 78616D70 6C65036F 72670002 6E730765 78616D70 6C65036F 72670000 01000100 000E1000 04C0A801 32
------------------------------
Length of DNSSEC Signed Data: 61 bytes
──────────────────────────────────────────────────


## 2. Public Key

In this demo, it is assumed that the verifier has access to the signer's public key. The public key is later used to verify the ladder's signature in Section 4.1. 

In [26]:
print("─" * 50)
print(f"Public Key Length: {len(pubkey)} bytes")
print("-" * 30)
print(f"Public Key Hex: {pubkey.hex().upper()}")
print("─" * 50)

──────────────────────────────────────────────────
Public Key Length: 1312 bytes
------------------------------
Public Key Hex: AF6E417BC8729F372F2CF5EE02EC25D43DC296E692B916EFB066E9A7941F22DFA156C3A9AF9C1AC316A625BF79D2F9D6B673686F2BBC263DE581853AD9FEF325B1B7E833BD30D0C11B52059FE889CFA6F0B6CB5A4EB608AE0C19F89F5194BDDC78C6F920046A3325E85F4A6A3ADF282DC9D5E122054E0DDC90A671252ABC96C5BAE9F4CF7FFCD6A2A6BC6315C601DC68F5199F8F37DB7DDCEF38CE2708C18E3491A96975E845C1FD45AA02C1FDA76C16799C59CE0EE2BDB4ED4455B600B70198F903C73EACE1908783B97EA1ECD845006E8D91DBE6512AA62CA413ED99E00F4638EBD9116E339ECB74BB500C3D725D94189E764FCF644F83F1B54EF33EEB10A4E0C15B8D5E4A3E1C7877A564A89E677795D637BA21CC39BB264461B0CADD986B43C20D5C5D71137F24019B5E1D37D210B6785D1BF07FCF896C0506E8A3477778FC6CD27FFB0126F3630402F47F756A1576D13ADEA08D6D04493B6FFCEA46C096E6FD4D37D8ABBB202301E62573DA1673B89211190DDD1F0925BEBA29F419752BBCA4E9C8612382C0D61780D0D2A7169434033E734F3FE4DF811E765581888473189443269023E55BCC916D40BA5E5117113333EC

## 3. Parsing Full Signature

In this demo, it is assumed that the signature is generated by the signer using the private key, and is delivered to the verifier. The full signature shown below **includes the authenticaton path, the ladder, and the underlying ladder signature**. 

In [27]:
print("─" * 50)
print(f"Full Signature Length: {len(fullsig)} bytes")
print("─" * 50)

──────────────────────────────────────────────────
Full Signature Length: 2649 bytes
──────────────────────────────────────────────────


The verifier then parses the full signature to obtain the authentication path, the ladder, and the ladder signature. 

```
+-- [Parsing Full Signature]
    +-- [(Section 3.1) Determine the MTL-Type]
    +-- [(Section 3.2) Obtianing Authentication Path]
    +-- [(Section 3.3) Obtaining Ladder and Underlying Ladder Signature]
    +-- [(Section 3.4) Visualizing MTL Structure]
```

#### Determine the MTL-Type
The signature starts with:
  * MTL-Type (1 byte): which should aways be hex value 0x01

In [28]:
from helper.mtl_parser import parse_mtl_type

mtl_type = parse_mtl_type(fullsig=fullsig, n=n)

──────────────────────────────────────────────────
MTL-Type: 01    - 1 byte
──────────────────────────────────────────────────


### 3.2 Obtaining Authentication Path (Merkle Tree Proof)

An authentication path is the Merkle tree proof of an MTL that can be used to verify a signed message. This message can be authenticated by recomputing leaf node from the message, recomputing internal nodes from the sibling node hash values obtained from the authentication path and previously computed hash values, and then comparing the result to the target rung hash value.

An authentication path consists of: 

```
                     1 1 1 1 1 1
 0 1 2 3 4 5 6 7 8 9 0 1 2 3 4 5
+-------------------------------+
|             Flags             |
+-------------------------------+
|                               |
//             SID             //
|                               |
+-------------------------------+
|                               |
//          Randomizer         //
|                               |
+-------------------------------+
|                               |
|           Leaf Index          |
|                               |
|                               |
+-------------------------------+
|                               |
|     Target Rung Left Index    |
|                               |
|                               |
+-------------------------------+
|                               |
|    Target Rung Right Index    |
|                               |
|                               |
+-------------------------------+
|       Sibling Hash Count      |
+-------------------------------+
|                               |
//      Sibling Hash Values    //
|                               |
+-------------------------------+
```

The AuthPath class below builds the AuthPath instance and initiates the instance with values extracted from the signature. Specifically, sibling indexes are calculated based on the leaf node index and the number of siblings.

In [29]:
from helper.mtl_parser import AuthPath

authpath = AuthPath(fullsig=fullsig, n=n)
authpath.authpath_print()

──────────────────────────────────────────────────
Authentication Path:
------------------------------
  Authentication Flags: 0000    - 2 bytes
------------------------------
  Series Identifier SID: 8F5CE818 A1DF091D FB8288CC AD6FF450 99A48C46 4F540CAB 05114981 736BB9D0    - 32 bytes
------------------------------
  Randomizer: C328BBA5 0C17738B 1F081ECE 4C0DB4C7    - 16 bytes
------------------------------
  Leaf Index: 00000000 00000004    - 8 bytes
------------------------------
  Target Rung Left Index: 00000000 00000000    - 8 bytes
------------------------------
  Target Rung Right Index: 00000000 00000007    - 8 bytes
------------------------------
  Sibling Hash Count: 3
------------------------------
  Sibling Node Hash Values:
    B1A6E1A4 DB73CE8D 9AC4449A B8F337B3    - 16 bytes
    5C91BF49 AF541835 32CD0584 162D7F8A    - 16 bytes
    D8D1F801 04C99E84 38ACF89A 8872237C    - 16 bytes
──────────────────────────────────────────────────
Level 0 Sibling:
  Left Index: 5
  Rig

### 3.3 Obtaining Ladder and Ladder Signature

A ladder is a collection of one or more rungs that can be used to verify an authentication path. Each node in the ladder is called a rung. A rung is a node from a node set that can be used to authenticate one or more leaf nodes within that node set. 

A signed ladder data structure consists of:

```
                     1 1 1 1 1 1
 0 1 2 3 4 5 6 7 8 9 0 1 2 3 4 5
+-------------------------------+
|             Flag              |
+-------------------------------+
|                               |
//             SID             //
|                               |
+-------------------------------+
|           Rung Count          |
+-------------------------------+
|                               |
//          Rung Data          //
|                               |
+-------------------------------+
|  Underlying Signature Length  |
|                               |
+-------------------------------+
|                               |
//     Underlying Signature    //
|                               |
+-------------------------------+
```

The Ladder class below builds the Ladder instance and initiates the instance with values extracted from the signature. Note that the SID extracted from the ladder should be consistent with the one extracted in Section 3.1. 

#### Ladder

In [30]:
from helper.mtl_parser import Ladder

ladder = Ladder(fullsig=fullsig, n=n, authpath_instance=authpath)
ladder.ladder_print()

──────────────────────────────────────────────────
Ladder:
------------------------------
  Ladder Flags: 0000    - 2 bytes
------------------------------
  Ladder Series Identifier SID: 8F5CE818 A1DF091D FB8288CC AD6FF450 99A48C46 4F540CAB 05114981 736BB9D0    - 32 bytes
------------------------------
  Ladder Rung Count: 2
------------------------------
  Ladder Rung:
------------------------------
    Rung Left Index: 00000000 00000000
    Rung Right Index: 00000000 00000007
    Rung Hash Value: FC1116C6 9A14E5AB 88ED3680 BDAC01FC
------------------------------
    Rung Left Index: 00000000 00000008
    Rung Right Index: 00000000 00000009
    Rung Hash Value: 4D846CCC C4586050 9E1A8189 0F3955CA
──────────────────────────────────────────────────


#### Ladder Signature

A ladder signature is generated by signing the ladder with the signer's private key. Verifiers then use the signer's public key to validate this signature, thereby verifying the ladder itself.

In [31]:
print("─" * 50)
# The length of the ladder signature is encoded in the ladder (uncomment the following to print)
print(f"Underlying Ladder Signature Length - {len(ladder.signature)} bytes")
print("─" * 50)

──────────────────────────────────────────────────
Underlying Ladder Signature Length - 2420 bytes
──────────────────────────────────────────────────


### 3.4 Visualizing MTL Structure

Based on extracted values from the signature, we can now visualize the MTL structure. We achieve this by: 
  * Calculating the total number of leaf nodes.
  * Building the list of all rung indexes. 
  * Calculating levels of nodes for the MTL.
  * Drawing an ASCII-art tree for visualization.  

In [32]:
from helper.utils import calculate_total_leaves, get_rung_index, generate_merkle_ladder, format_tree_ascii

total_leaves = calculate_total_leaves(ladder.rungs)
all_rungs = get_rung_index(ladder.rungs)
ladder_layers = generate_merkle_ladder(total_leaves, all_rungs)
print(format_tree_ascii(ladder_layers))

                     (0, 7)*
                        |
                        H
            /----------- -----------\
         (0, 3)                  (4, 7)
            |                       |
            H                       H
      /----- -----\           /----- -----\
   (0, 1)      (2, 3)      (4, 5)      (6, 7)      (8, 9)*
      |           |           |           |           |
      H           H           H           H           H
    /- -\       /- -\       /- -\       /- -\       /- -\
   0     1     2     3     4     5     6     7     8     9


## 4. Verification

The verification operation of a full signature involves two main steps:
* Verifying the signature on the signed ladder using the signer's public key. 
* Verifying the authentication path of the message relative to the ladder.

The verification pipeline is as follows:
```
+-- [Verification]
    +-- [(Section 4.1) - Validating Ladder Signature Using Public Key]
    +-- [(Section 4.2) - Validating Authentication Path]
    |   +-- [(Section 4.2.1) Recomputing Leaf Hash]
    |   +-- [(Section 4.2.2) Iteratively Hashing with Sibling Nodes for Internal Nodes]
    |   +-- [(Section 4.2.3) Target Match with Extracted Ladder Rung]
```

### 4.1 Validating Ladder Signature Using Signer's Public Key

#### Helper: check OpenSSL version

In [33]:
# Load the shared library
libcrypto = ctypes.CDLL("/usr/local/lib64/libcrypto.so.3")

# OpenSSL version: 0 maps to the OPENSSL_VERSION string constant
libcrypto.OpenSSL_version.restype = ctypes.c_char_p
version_text = libcrypto.OpenSSL_version(0) 

print(version_text.decode("utf-8"))

OpenSSL 3.5.0 8 Apr 2025


We configure OpenSSL parameters and map handlers to Python ctypes in the helper file verifier.py. We bind OpenSSL to validate the ladder signature.

#### Validating Ladder Signature

In [34]:
from helper.verifier import mldsa_verify_signature, slhdsa_verify_signature

def verify_signature(algo: str, pubkey: bytes, signature: bytes, message: bytes) -> bool:
    if algo.startswith("ML-DSA"):
        is_verified = mldsa_verify_signature(algorithm=algo, public_key=pubkey, ladder_sig=signature, ladder=message)
    else:
        raise ValueError(f"Unsupported algorithm {algo}.")
    return is_verified

In [35]:
is_verified = verify_signature(algo=sig_alg, pubkey=pubkey, signature=ladder.signature, message=ladder.val)
    
print("─" * 50)
print(f"Ladder Signature Verification Result: {'SUCCESS!' if is_verified else 'FAILED!!'}")
print("─" * 50)

──────────────────────────────────────────────────
Ladder Signature Verification Result: SUCCESS!
──────────────────────────────────────────────────


Now the ladder is verified. The next step is to verify the authentication path of the message relative to the ladder.

### 4.2 Validating Authentication Path

The verifier verifies the authentication path for a message relative to a rung. This verification operation has two main steps:
* First, hash_leaf() computes the leaf node hash value from the message and authentication path. If the leaf node index matches the rung index pair, the leaf hash is compared to the rung hash.
* Second, hash_int() computes internal node hash values using the leaf hash and successive sibling hashes from the authentication path. If any internal node index matches the rung index, its hash is verified against the rung hash.

#### 4.2.1 Leaf Node Hash
The function below constructs the leaf node payload and executes the hash operation using the specified algorithm and security parameter.

In [36]:
from helper.verifier import hash_leaf

# Leave the context string (ctx_msg) NULL
leaf_node = hash_leaf(
                    sid=authpath.sid,
                    leaf_index=int(authpath.leaf_index.hex(), 16),
                    randomizer=authpath.randomizer,
                    ctx_msg=b"",
                    message=message,
                    hash_alg=hash_alg,
                    n=n
)

# Update AuthPath instance
authpath.leaf_node = leaf_node

print("─" * 50)
print(f"Leaf Node ({int(authpath.leaf_index.hex(), 16)}, {int(authpath.leaf_index.hex(), 16)}): {authpath.leaf_node.hex(" ", -4).upper()}")
print(f"Length of Leaf Node: {len(leaf_node)} bytes")
print("─" * 50)

──────────────────────────────────────────────────
Leaf Node (4, 4): CEB61A79 CEFAFC18 BE08495D 1D9F212D
Length of Leaf Node: 16 bytes
──────────────────────────────────────────────────


#### 4.2.2 Iteratively Hashing with Sibling Nodes to Obtain Internal Nodes

We next iteratively hash with sibling nodes by performing the following for each sibliing node: 
  * Generate the hash payload.
  * Perform the hash operation.

In [37]:
from helper.verifier import calculate_internal_nodes

internal_nodes = calculate_internal_nodes(sid=authpath.sid, 
                                          leaf_index=authpath.leaf_index, 
                                          leaf_hash=authpath.leaf_node, 
                                          sibling_list=authpath.siblings_withindex, 
                                          hash_alg=hash_alg, 
                                          n=n)

# Update AuthPath instance
authpath.internal_nodes = internal_nodes

print("─" * 50)
for i in range(len(internal_nodes)):
    curr = internal_nodes[i]
    print(f"Internal Node ({curr["left_index"]}, {curr["right_index"]}): {curr["hash"].hex(" ", -4).upper()}")
    if i != len(internal_nodes) - 1:
        print("-" * 50)
print("─" * 50)

──────────────────────────────────────────────────
Internal Node (4, 5): 439EA68B 0BF8C252 782CE893 4BBEDAD2
--------------------------------------------------
Internal Node (4, 7): 2EF001D1 58EF395C B7AF58BA FBEEBB31
--------------------------------------------------
Internal Node (0, 7): FC1116C6 9A14E5AB 88ED3680 BDAC01FC
──────────────────────────────────────────────────


#### 4.2.3 Target Match with Extracted Ladder Rung 

Finally, the computed root internal node is compared to the target rung hash from the verified ladder to validate the authentication path. 

In [38]:
from helper.verifier import compare_with_rung

ret = compare_with_rung(internal_nodes=authpath.internal_nodes,
                        target_rung_lindex=authpath.rung_lindex,
                        target_rung_rindex=authpath.rung_rindex,
                        rungs=ladder.rungs)

print("─" * 50)
if ret:
    print("Verification SUCCESS!")
else:
    print("Verificaton FAILED!!!")
print("─" * 50)

──────────────────────────────────────────────────
Verification SUCCESS!
──────────────────────────────────────────────────


## End of Demo

This demo shows how an ML-DSA signature for the MTL Mode is parsed into components and verified by:
  * Validating the signed ladder using the corresponding public key, and
  * Verifying the message authentication path relative to a specific rung.